# Jensen's inequality

_Probability & Statistics_

**Average a curved-up function and you get more than the function of the average.**

A function is convex if its graph curves up &mdash; it lies below every straight line drawn between two of its points (a chord), and above every tangent line that just touches it. Think of the bowl shape of $f(x) = x^2$.

       A function is concave if it curves the other way (down) &mdash; the upside-down bowl. The $\log$ function is the standard example.

---

This notebook is a practice scaffold for the **Jensen's inequality** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Set up samples and a Jensen-gap helperJensen's inequality compares two numbers: **E[f(X)]** (transform every sample, *then* average) versus **f(E[X])** (average the samples first, *then* transform once). We draw a large sample so the averages are accurate, and write one helper that prints both quantities and their gap E[f(X)] − f(E[X]).

In [ ]:
rng = np.random.default_rng(0)
N = 100_000

def jensen_gap(f, X, name):
    Ef = np.mean(f(X))      # E[f(X)] : transform every sample, then average
    fE = f(np.mean(X))      # f(E[X]) : average first, then transform once
    print(f"{name:24s}  E[f(X)]={Ef:8.4f}   f(E[X])={fE:8.4f}   gap={Ef - fE:+.4f}")
    return Ef - fE

### Step 2 — Convex case: the gap is positive and shrinks with spreadFor a **convex** f (here f(x) = eˣ), Jensen guarantees E[f(X)] ≥ f(E[X]), so the gap is ≥ 0. The size of the gap depends on how spread out X is: as the standard deviation σ falls toward 0, X concentrates at its mean and the gap shrinks toward 0.

In [ ]:
# CONVEX f(x) = e^x : Jensen says E[f(X)] >= f(E[X]), so gap >= 0
print("convex f(x) = exp(x),  X ~ Normal(0, sigma):  gap should be >= 0 and shrink with sigma")

for sigma in [1.0, 0.5, 0.25, 0.0]:
    if sigma > 0:
        X = rng.normal(0.0, sigma, N)
    else:
        X = np.zeros(N)            # zero spread: every sample sits exactly at the mean
    jensen_gap(np.exp, X, f"  std(X) = {sigma:.2f}")

### Step 3 — Concave case: the inequality flipsFor a **concave** f (here f(x) = log x), the inequality reverses: E[f(X)] ≤ f(E[X]), so the gap is ≤ 0. We draw X from Uniform(1, 5) — staying positive so the log is defined — and confirm that averaging the logs lands *below* the log of the average.

In [ ]:
# CONCAVE f(x) = log x : the inequality FLIPS, so gap <= 0
print("\nconcave f(x) = log(x),  X ~ Uniform(1, 5):  gap should be <= 0")

Xpos = rng.uniform(1.0, 5.0, N)
jensen_gap(np.log, Xpos, "  uniform(1, 5)")

## Visualize the data & results

_For a convex function the average overshoots f(E[X]); for a concave one it undershoots. How big is the gap, and which way does it point?_

### Step 1 — Recompute the convex case for plottingTo visualise the gap we recompute the convex example directly (rather than through the helper) so we have the two numbers in hand. With X ~ Normal(0, 1) the sample mean is ≈ 0, so f(E[X]) ≈ e⁰ ≈ 1, while E[eˣ] sits well above it — a clearly positive gap.

In [ ]:
rng = np.random.default_rng(0)
N = 100_000

# CONVEX f(x) = e^x, X ~ Normal(0, 1):  E[f(X)] >= f(E[X])
X = rng.normal(0.0, 1.0, N)
conv_Ef = np.mean(np.exp(X))     # 1.6482
conv_fE = np.exp(np.mean(X))     # 0.9991  (mean(X) ~ 0, so e^0 ~ 1)

### Step 2 — Recompute the concave caseNow the concave example with X ~ Uniform(1, 5). Here the relationship flips: the average of the logs (E[log X]) falls *below* the log of the average (log E[X]), giving a negative gap.

In [ ]:
# CONCAVE f(x) = log x, X ~ Uniform(1, 5):  E[f(X)] <= f(E[X])
Xp = rng.uniform(1.0, 5.0, N)
conc_Ef = np.mean(np.log(Xp))    # 1.0141
conc_fE = np.log(np.mean(Xp))    # 1.1010

### Step 3 — Compare the two pairsPrinting both pairs side by side makes the two directions concrete: the convex pair shows E[f(X)] above f(E[X]) (gap +0.65), and the concave pair shows it below (gap −0.09).

In [ ]:
print(round(conv_Ef, 4), round(conv_fE, 4))   # 1.6482 0.9991  -> gap +0.6491
print(round(conc_Ef, 4), round(conc_fE, 4))   # 1.0141 1.101   -> gap -0.0869

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Let $X$ be $2$ or $4$ each with probability $\tfrac{1}{2}$, and $f(x) = x^2$. Compute $E[f(X)]$ and $f(E[X])$, confirm Jensen, and identify the gap.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean: $E[X] = \tfrac{2+4}{2} = 3$, so $f(E[X]) = 3^2 = 9$. — _"Average first, then square" is the right-hand side of the convex bound._
- Transformed mean: $E[X^2] = \tfrac{2^2 + 4^2}{2} = \tfrac{4 + 16}{2} = 10$. — _"Square first, then average" is the left-hand side._
- Compare: $10 \ge 9$, so Jensen holds. — _$x^2$ is convex, so $E[f(X)] \ge f(E[X])$ is guaranteed._

**Answer:** $E[f(X)] = 10$, $f(E[X]) = 9$, gap $= 1$, which equals $\operatorname{Var}(X) = (\pm 1)^2 = 1$.

</details>

**Problem 2.** For a concave function such as $f(x) = \log x$, which way does Jensen point, and what does that say about $E[\log X]$ versus $\log E[X]$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\log$ curves downward, so it is concave. — _Concave means the curve lies below its chords and the inequality reverses._
- Apply the concave form $E[f(X)] \le f(E[X])$. — _Concave flips the convex $\ge$ into $\le$._

**Answer:** $E[\log X] \le \log E[X]$. Averaging logs gives at most the log of the average. This is the exact step that makes the EM lower bound (ELBO) and the KL divergence behave, and it is why $\log E[\cdot] \ge E[\log \cdot]$.

</details>